In [1]:
import sportsipy

In [4]:
"""
Build a college football dataset from Sportsipy.

What this script does:
- Loops over a range of seasons
- For each season, pulls all team season stats (sportsipy.ncaaf.teams.Teams)
- For each team, pulls its schedule (sportsipy.ncaaf.schedule.Schedule)
- Keeps only games where that team is the HOME team
- Merges in season-level stats for the home team and the opponent
- Adds home_win and point_diff targets
- Writes everything to a CSV

Requirements:
    pip install sportsipy pandas
"""

import pandas as pd
from sportsipy.ncaaf.teams import Teams


# CONFIG: change this to the seasons you care about

SEASONS = list(range(2002, 2024))   # e.g. 2015–2023
OUTPUT_CSV = "ncaaf_home_games_with_team_stats.csv"


def build_team_season_stats(seasons):
    """
    Build a DataFrame with one row per team-season using Team.dataframe.

    Columns include things like:
      - abbreviation, name, conference
      - points_per_game, points_against_per_game
      - strength_of_schedule, simple_rating_system
      - rush_yards, pass_yards, turnovers, etc.
    """
    all_rows = []

    for year in seasons:
        print(f"Pulling team season stats for {year}...")
        try:
            teams = Teams(year)
        except Exception as e:
            print(f"  ! Failed to load Teams({year}): {e}")
            continue

        for team in teams:
            try:
                df = team.dataframe  # 1-row DataFrame
                df = df.reset_index(drop=True)
                df["season"] = int(year)
                all_rows.append(df)
            except Exception as e:
                print(f"    ! Failed for team {getattr(team, 'abbreviation', 'UNKNOWN')} in {year}: {e}")

    if not all_rows:
        raise RuntimeError("No team season stats collected. Check seasons / network / sportsipy.")

    team_season_stats = pd.concat(all_rows, ignore_index=True)
    # Ensure we have 'abbreviation' as a column to join on
    if "abbreviation" not in team_season_stats.columns:
        raise RuntimeError("Expected 'abbreviation' column not found in team_season_stats.")

    return team_season_stats


def build_home_games_dataset(seasons, team_season_stats):
    """
    Build a DataFrame with one row per HOME game.

    Uses:
      - team.schedule.dataframe  (Game-level info for that team)
      - merges in season stats for home & away teams

    Output columns include:
      - season, datetime, day_of_week
      - team_abbr/team_name/team_conference (home team)
      - opponent_abbr/opponent_name/opponent_conference
      - location (always 'Home' in this dataset)
      - points_for, points_against, rank, opponent_rank, wins, losses, streak
      - home_win (binary), point_diff (home_points - away_points)
      - is_conference_game (1 if same conference)
      - home_* season stats
      - away_* season stats
    """
    all_games = []

    for year in seasons:
        print(f"Pulling schedules for {year}...")
        try:
            teams = Teams(year)
        except Exception as e:
            print(f"  ! Failed to load Teams({year}) for schedules: {e}")
            continue

        for team in teams:
            try:
                sched_df = team.schedule.dataframe  # DataFrame of Game properties
            except Exception as e:
                print(f"    ! Failed to get schedule for {getattr(team, 'abbreviation', 'UNKNOWN')} in {year}: {e}")
                continue

            if sched_df is None or sched_df.empty:
                continue

            # Index is boxscore string; keep it as a column
            sched_df = sched_df.reset_index().rename(columns={"index": "boxscore_index"})
            sched_df["season"] = int(year)
            sched_df["team_abbr"] = team.abbreviation
            sched_df["team_name"] = team.name
            sched_df["team_conference"] = team.conference

            # Keep only HOME games so each game appears once,
            # oriented from the actual home team's perspective.
            home_df = sched_df[sched_df["location"] == "Home"].copy()
            if home_df.empty:
                continue

            all_games.append(home_df)

    if not all_games:
        raise RuntimeError("No home games collected. Check seasons / network / sportsipy.")

    games = pd.concat(all_games, ignore_index=True)

    # Merge in season stats for home & away teams

    # Keys and features from the team season stats
    key_cols = ["season", "abbreviation"]
    feature_cols = [c for c in team_season_stats.columns if c not in key_cols]

    # Home team stats
    home_stats = team_season_stats[key_cols + feature_cols].copy()
    home_stats = home_stats.rename(
        columns={c: f"home_{c}" for c in feature_cols}
    )
    # Merge: home team is games.team_abbr
    games = games.merge(
        home_stats,
        how="left",
        left_on=["season", "team_abbr"],
        right_on=["season", "abbreviation"],
    ).drop(columns=["abbreviation"])

    # Away (opponent) team stats
    away_stats = team_season_stats[key_cols + feature_cols].copy()
    away_stats = away_stats.rename(
        columns={c: f"away_{c}" for c in feature_cols}
    )
    # Merge: away team is games.opponent_abbr
    games = games.merge(
        away_stats,
        how="left",
        left_on =["season", "opponent_abbr"],
        right_on=["season", "abbreviation"],
        suffixes=("", "_dup"),
    ).drop(columns=["abbreviation"])

    # Targets and helper flags
    # Rename points_for/against into home/away points
    games["home_points"] = games["points_for"]
    games["away_points"] =games["points_against"]
    games["point_diff"] = games["home_points"] - games["away_points"]

    # Binary outcome for logistic regression (home win vs not)
    games["home_win"] = (games["result"] == "W").astype(int)

    # Conference game flag (simple version: same conference string)
    games["is_conference_game"] = (
        games["team_conference"] == games["opponent_conference"]
    ).astype(int)

    return games


def main():
    # 1) Team-season stats
    team_season_stats = build_team_season_stats(SEASONS)
    print(f"Collected {len(team_season_stats)} team-season rows.")

    # 2) Game-level dataset (home games only)
    games = build_home_games_dataset(SEASONS, team_season_stats)
    print(f"Collected {len(games)} home games.")

    # 3) Write CSV
    games.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved dataset to {OUTPUT_CSV}")


if __name__ == "__main__":
    main()


/Users/will/Applications/miniconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Pulling team season stats for 2002...
The requested page returned a valid response, but no data could be found. Has the season begun, and is the data available on www.sports-reference.com?
Pulling team season stats for 2003...
The requested page returned a valid response, but no data could be found. Has the season begun, and is the data available on www.sports-reference.com?
Pulling team season stats for 2004...
The requested page returned a valid response, but no data could be found. Has the season begun, and is the data available on www.sports-reference.com?
Pulling team season stats for 2005...
The requested page returned a valid response, but no data could be found. Has the season begun, and is the data available on www.sports-reference.com?
Pulling team season stats for 2006...
The requested page returned a valid response, but no data could be found. Has the season begun, and is the data available on www.sports-reference.com?
Pulling team season stats for 2007...
The requested pag

RuntimeError: No team season stats collected. Check seasons / network / sportsipy.

1. Game-Level Identifiers & Meta
| Variable          | Type   | Description                                                                                                                     |
| ----------------- | ------ | ------------------------------------------------------------------------------------------------------------------------------- |
| `boxscore_index`  | string | Unique ID for the game (Sports-Reference boxscore slug). Can be used to link to boxscore data or merge with other data sources. |
| `season`          | int    | Season year in which the game was played (e.g., `2019`).                                                                        |
| `team_abbr`       | string | Abbreviation of the **home team** (e.g., `PURDUE`).                                                                             |
| `team_name`       | string | Full name of the home team (e.g., `Purdue Boilermakers`).                                                                       |
| `team_conference` | string | Conference of the home team (e.g., `big-10`).                                                                                   |
2. Game Context & Schedule Fields
| Variable      | Type   | Description                                                                                              |
| ------------- | ------ | -------------------------------------------------------------------------------------------------------- |
| `date`        | string | Human-readable game date (e.g., `"Sep 2, 2017"`).                                                        |
| `datetime`    | string | Full date and kickoff time in ISO-like format (e.g., `"2017-09-02 19:30:00"`).                           |
| `day_of_week` | string | Day of week abbreviation (e.g., `Mon`, `Sat`).                                                           |
| `game`        | int    | Game number in the team’s season (1 = first game, etc.).                                                 |
| `location`    | string | Game location relative to this team (`"Home"`, `"Away"`, `"Neutral"`). In this dataset, always `"Home"`. |
| `time`        | string | Kickoff time string as listed on Sports-Reference.                                                       |

3. Opponent Fields
| Variable              | Type      | Description                                                      |
| --------------------- | --------- | ---------------------------------------------------------------- |
| `opponent_abbr`       | string    | Abbreviation of the **away team** (opponent).                    |
| `opponent_name`       | string    | Full name of the opponent team.                                  |
| `opponent_conference` | string    | Opponent’s conference. Non-FBS teams usually have a special tag. |
| `opponent_rank`       | int / NaN | Rank of the opponent at game time; `NaN` if unranked.            |


4. Result & Running Record
| Variable         | Type      | Description                                                       |
| ---------------- | --------- | ----------------------------------------------------------------- |
| `points_for`     | int       | Points scored by the **home team** in this game.                  |
| `points_against` | int       | Points scored by the **away team** in this game.                  |
| `rank`           | int / NaN | Rank of the home team at the time of the game; `NaN` if unranked. |
| `result`         | string    | Game result from the home team’s perspective: `"W"` or `"L"`.     |
| `streak`         | string    | Win/loss streak after this game (e.g., `"W 3"`, `"L 2"`).         |
| `wins`           | int       | Cumulative season wins for the home team after this game.         |
| `losses`         | int       | Cumulative season losses for the home team after this game.       |


5. Engineered Targets & Flags
| Variable             | Type      | Description                                                                                                                 |
| -------------------- | --------- | --------------------------------------------------------------------------------------------------------------------------- |
| `home_points`        | int       | Alias for `points_for` (home team points).                                                                                  |
| `away_points`        | int       | Alias for `points_against` (away team points).                                                                              |
| `point_diff`         | int       | `home_points - away_points`. Positive if the home team wins by that many points; negative if they lose by that many points. |
| `home_win`           | int (0/1) | Binary outcome: `1` if `result == "W"`, else `0`. Used as the target for logistic regression.                               |
| `is_conference_game` | int (0/1) | `1` if `team_conference == opponent_conference`, else `0`. Simple indicator for conference games.                           |


6. Season-Level Team Stats (Home / Away Prefixes)
The script merges in season-level statistics from Team.dataframe twice:
- once for the home team (prefix home_)
- once for the away team / opponent (prefix away_)
So for each base team stat called <stat>, you will have:
- home_<stat> — value of <stat> for the home team in that season
- away_<stat> — value of <stat> for the away team in that season
Below are some of the key stats likely to use.
- 6.1 Identification & Outcome Stats
| Variable pattern                                                   | Description                                        |
| ------------------------------------------------------------------ | -------------------------------------------------- |
| `home_abbreviation`, `away_abbreviation`                           | Team abbreviations.                                |
| `home_name`, `away_name`                                           | Full team names.                                   |
| `home_conference`, `away_conference`                               | Team conferences (same as above but season-level). |
| `home_games`, `away_games`                                         | Number of games played in that season.             |
| `home_wins`, `away_wins`                                           | Total wins that season.                            |
| `home_losses`, `away_losses`                                       | Total losses that season.                          |
| `home_win_percentage`, `away_win_percentage`                       | Season win percentage.                             |
| `home_conference_wins`, `away_conference_wins`                     | Conference wins.                                   |
| `home_conference_losses`, `away_conference_losses`                 | Conference losses.                                 |
| `home_conference_win_percentage`, `away_conference_win_percentage` | Conference win percentage.                         |

- 6.2 Team Strength Metrics
| Variable pattern                                               | Description                                                                                   |
| -------------------------------------------------------------- | --------------------------------------------------------------------------------------------- |
| `home_points_per_game`, `away_points_per_game`                 | Average points scored per game.                                                               |
| `home_points_against_per_game`, `away_points_against_per_game` | Average points allowed per game.                                                              |
| `home_simple_rating_system`, `away_simple_rating_system`       | Overall strength rating: margin of victory adjusted for schedule strength. Higher = stronger. |
| `home_strength_of_schedule`, `away_strength_of_schedule`       | Strength of schedule in points above/below average. Positive = tougher schedule.              |

- 6.3 Offensive Season Stats (Per Game)
| Variable pattern               | Description                                              |
| ------------------------------ | -------------------------------------------------------- |
| `*_yards`                      | Total offensive yards per game.                          |
| `*_yards_per_play`             | Offensive yards per play.                                |
| `*_plays`                      | Offensive plays per game.                                |
| `*_first_downs`                | Offensive first downs per game.                          |
| `*_pass_attempts`              | Pass attempts per game.                                  |
| `*_pass_completions`           | Pass completions per game.                               |
| `*_pass_completion_percentage` | Completion percentage.                                   |
| `*_pass_yards`                 | Passing yards per game.                                  |
| `*_pass_touchdowns`            | Passing TDs per game.                                    |
| `*_rush_attempts`              | Rush attempts per game.                                  |
| `*_rush_yards`                 | Rushing yards per game.                                  |
| `*_rush_yards_per_attempt`     | Yards per rush attempt.                                  |
| `*_rush_touchdowns`            | Rushing TDs per game.                                    |
| `*_penalties`                  | Penalties committed per game.                            |
| `*_yards_from_penalties`       | Yards gained from opponent penalties per game.           |
| `*_turnovers`                  | Total turnovers per game (fumbles lost + interceptions). |
| `*_fumbles_lost`               | Fumbles lost per game.                                   |
| `*_interceptions`              | Interceptions thrown per game.                           |

- 6.4 Defensive / Opponent Season Stats
| Variable pattern                   | Description                                     |
| ---------------------------------- | ----------------------------------------------- |
| `*_opponents_yards`                | Yards allowed per game.                         |
| `*_opponents_yards_per_play`       | Yards allowed per play.                         |
| `*_opponents_pass_yards`           | Passing yards allowed per game.                 |
| `*_opponents_pass_touchdowns`      | Passing TDs allowed per game.                   |
| `*_opponents_rush_yards`           | Rushing yards allowed per game.                 |
| `*_opponents_rush_touchdowns`      | Rushing TDs allowed per game.                   |
| `*_opponents_first_downs`          | First downs allowed per game.                   |
| `*_opponents_turnovers`            | Turnovers forced per game (takeaways).          |
| `*_opponents_penalties`            | Opponent penalties per game.                    |
| `*_opponents_yards_from_penalties` | Yards conceded via opponent penalties per game. |


7. Typical Features You’ll Use (for quick reference)
When building models, you’ll probably create differences or derived features:
- srs_diff = home_simple_rating_system - away_simple_rating_system
- off_ppg_diff = home_points_per_game - away_points_per_game
- def_ppg_diff = away_points_against_per_game - home_points_against_per_game
- win_pct_diff = home_win_percentage - away_win_percentage
with:
- home_win as the binary target
- point_diff as the regression target

In [10]:
import kagglehub
import os
import shutil

TARGET_DIR = r"/Users/will/GitHub/IS477/data/raw"  

# Download latest version (goes into KaggleHub cache)
path = kagglehub.dataset_download(
    "cviaxmiwnptr/college-football-team-stats-2002-to-january-2024"
)

print("KaggleHub cache path:", path)

os.makedirs(TARGET_DIR, exist_ok=True)

# Copy all files
for filename in os.listdir(path):
    src = os.path.join(path, filename)
    dst = os.path.join(TARGET_DIR, filename)

    # If it's a file, copy it
    if os.path.isfile(src):
        shutil.copy2(src, dst)
        print(f"Copied {src} -> {dst}")

print("\nAll dataset files copied to:", TARGET_DIR)


KaggleHub cache path: /Users/will/.cache/kagglehub/datasets/cviaxmiwnptr/college-football-team-stats-2002-to-january-2024/versions/3
Copied /Users/will/.cache/kagglehub/datasets/cviaxmiwnptr/college-football-team-stats-2002-to-january-2024/versions/3/cfb_box-scores_2002-2024.csv -> /Users/will/GitHub/IS477/data/raw/cfb_box-scores_2002-2024.csv

All dataset files copied to: /Users/will/GitHub/IS477/data/raw


In [12]:
df = pd.read_csv("/Users/will/GitHub/IS477/data/raw/cfb_box-scores_2002-2024.csv")

In [15]:
df.columns

Index(['season', 'week', 'date', 'time_et', 'game_type', 'away', 'home',
       'rank_away', 'rank_home', 'conf_away', 'conf_home', 'neutral',
       'score_away', 'score_home', 'q1_away', 'q2_away', 'q3_away', 'q4_away',
       'ot_away', 'q1_home', 'q2_home', 'q3_home', 'q4_home', 'ot_home',
       'first_downs_away', 'first_downs_home', 'third_down_comp_away',
       'third_down_att_away', 'third_down_comp_home', 'third_down_att_home',
       'fourth_down_comp_away', 'fourth_down_att_away',
       'fourth_down_comp_home', 'fourth_down_att_home', 'pass_comp_away',
       'pass_att_away', 'pass_yards_away', 'pass_comp_home', 'pass_att_home',
       'pass_yards_home', 'rush_att_away', 'rush_yards_away', 'rush_att_home',
       'rush_yards_home', 'total_yards_away', 'total_yards_home', 'fum_away',
       'fum_home', 'int_away', 'int_home', 'pen_num_away', 'pen_yards_away',
       'pen_num_home', 'pen_yards_home', 'possession_away', 'possession_home',
       'attendance', 'tv'],
      dt